# Week 1 Lab: Data Collection for Machine Learning

**CS 203: Software Tools and Techniques for AI**

---

## Lab Overview

In this lab, you will learn to collect data from the web using:

1. **HTTP fundamentals** - Understanding how the web works
2. **curl** - Command-line HTTP client
3. **Python requests** - Programmatic API calls
4. **BeautifulSoup** - Web scraping when APIs don't exist

**Goal**: Build a movie data collection pipeline for Netflix-style movie prediction.

---

## Setup

First, let's install and import the required libraries.

In [ ]:
# Install required packages (uncomment if needed)
# !pip install requests beautifulsoup4 pandas

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import time

print("All imports successful!")

---

# Part 1: HTTP Fundamentals

Before we start collecting data, we need to understand how the web works.

## 1.1 Understanding URLs

A URL (Uniform Resource Locator) has several components:

```
https://api.omdbapi.com:443/v1/movies?t=Inception&y=2010#details
└─┬──┘ └──────┬───────┘└┬─┘└───┬───┘└─────────┬────────┘└───┬───┘
  │           │         │      │              │             │
Protocol    Host      Port   Path          Query        Fragment
```

### Question 1.1 (Solved): Parse a URL

Use Python's `urllib.parse` to break down a URL into its components.

In [ ]:
# SOLVED EXAMPLE
from urllib.parse import urlparse, parse_qs

url = "https://api.omdbapi.com/?apikey=demo&t=Inception&y=2010"

parsed = urlparse(url)

print(f"Scheme (protocol): {parsed.scheme}")
print(f"Host (domain): {parsed.netloc}")
print(f"Path: {parsed.path}")
print(f"Query string: {parsed.query}")

# Parse query parameters into a dictionary
params = parse_qs(parsed.query)
print(f"\nParsed parameters: {params}")

### Question 1.2: Parse a Different URL

Parse the following GitHub API URL and extract:
1. The host
2. The path
3. All query parameters as a dictionary

URL: `https://api.github.com/search/repositories?q=machine+learning&sort=stars&order=desc`

In [ ]:
# YOUR CODE HERE
url = "https://api.github.com/search/repositories?q=machine+learning&sort=stars&order=desc"

from urllib.parse import urlparse, parse_qs

# Parse the URL
parsed = urlparse(url)

# Print the host
print(f"Host: {parsed.netloc}")

# Print the path
print(f"Path: {parsed.path}")

# Print the query parameters as a dictionary
params = parse_qs(parsed.query)
print(f"Query parameters: {params}")


---

## 1.2 HTTP Status Codes

HTTP status codes tell you what happened with your request:

| Range | Category | Common Examples |
|-------|----------|----------------|
| 2xx | Success | 200 OK, 201 Created |
| 3xx | Redirect | 301 Moved, 302 Found |
| 4xx | Client Error | 400 Bad Request, 401 Unauthorized, 404 Not Found |
| 5xx | Server Error | 500 Internal Error, 503 Service Unavailable |

### Question 1.3: Match Status Codes

Match each scenario to the most likely HTTP status code:

1. You requested a movie that doesn't exist in the database
2. You made too many requests and hit the rate limit
3. Your API key is invalid
4. The request was successful and data was returned
5. The server crashed while processing your request

Status codes to choose from: `200`, `401`, `404`, `429`, `500`

In [ ]:
# YOUR ANSWERS HERE
answers = {
    "movie_not_found": 404,   # Resource not found
    "rate_limited": 429,       # Too Many Requests
    "invalid_api_key": 401,    # Unauthorized
    "success": 200,            # OK
    "server_crashed": 500      # Internal Server Error
}

print(answers)


---

# Part 2: Making Requests with `curl`

`curl` is a command-line tool for making HTTP requests. It's essential for quick testing.

## 2.1 Basic curl Commands

You can run shell commands in Jupyter using `!` prefix.

### Question 2.1 (Solved): Your First API Call

Let's call a simple public API that requires no authentication.

In [ ]:
# SOLVED EXAMPLE
# JSONPlaceholder is a free fake API for testing
!curl -s "https://jsonplaceholder.typicode.com/posts/1"

### Question 2.2: Pretty Print with jq

The output above is hard to read. Use `jq` to format it nicely.

**Hint**: Pipe the curl output to jq: `curl ... | jq .`

In [ ]:
# YOUR CODE HERE
# Fetch the same post but format the output with jq
!curl -s "https://jsonplaceholder.typicode.com/posts/1" | jq .


### Question 2.3: Extract Specific Fields with jq

Fetch all posts from `https://jsonplaceholder.typicode.com/posts` and extract only the `title` field from each post.

**Hint**: Use `jq '.[].title'` to get the title from each element in the array.

In [ ]:
# YOUR CODE HERE
# Fetch all posts and extract only the 'title' field from each
!curl -s "https://jsonplaceholder.typicode.com/posts" | jq '.[].title'


### Question 2.4: View Response Headers

Use the `-I` flag to fetch only the response headers (no body) from:
`https://api.github.com`

What is the value of the `X-RateLimit-Limit` header?

In [ ]:
# YOUR CODE HERE
# Use -I flag to fetch only response headers from GitHub API
!curl -sI "https://api.github.com"


### Question 2.5: Add Custom Headers

Make a request to `https://httpbin.org/headers` with the following custom headers:
- `User-Agent: CS203-Lab/1.0`
- `Accept: application/json`

**Hint**: Use `-H "Header-Name: value"` for each header.

In [ ]:
# YOUR CODE HERE
# Make a request to httpbin.org/headers with custom headers
!curl -s "https://httpbin.org/headers" \
    -H "User-Agent: CS203-Lab/1.0" \
    -H "Accept: application/json"


---

# Part 3: Python `requests` Library

While `curl` is great for testing, we need Python for automation.

## 3.1 Basic GET Requests

### Question 3.1 (Solved): Simple GET Request

Make a GET request and inspect the response object.

In [ ]:
# SOLVED EXAMPLE
import requests

response = requests.get("https://jsonplaceholder.typicode.com/posts/1")

print(f"Status Code: {response.status_code}")
print(f"Content-Type: {response.headers['Content-Type']}")
print(f"Response OK: {response.ok}")
print(f"\nJSON Data:")
print(response.json())

### Question 3.2: Fetch Multiple Posts

Fetch posts from `https://jsonplaceholder.typicode.com/posts` and:
1. Print the total number of posts
2. Print the titles of the first 5 posts

In [ ]:
# YOUR CODE HERE
import requests

# Fetch all posts
response = requests.get("https://jsonplaceholder.typicode.com/posts")
posts = response.json()

# 1. Total number of posts
print(f"Total number of posts: {len(posts)}")

# 2. Titles of the first 5 posts
print("\nFirst 5 post titles:")
for post in posts[:5]:
    print(f"  [{post['id']}] {post['title']}")


### Question 3.3 (Solved): Using Query Parameters

The proper way to add query parameters is using the `params` argument.

In [ ]:
# SOLVED EXAMPLE
import requests

# Bad way (manual string building)
# url = "https://jsonplaceholder.typicode.com/posts?userId=1"

# Good way (using params)
response = requests.get(
    "https://jsonplaceholder.typicode.com/posts",
    params={"userId": 1}
)

posts = response.json()
print(f"User 1 has {len(posts)} posts")
print(f"\nActual URL used: {response.url}")

### Question 3.4: Filter Posts by User

Fetch all posts by user 5 and user 7. Compare how many posts each user has.

**Hint**: Make two separate requests with different `userId` values.

In [ ]:
# YOUR CODE HERE
import requests

# Fetch posts for user 5
resp5 = requests.get(
    "https://jsonplaceholder.typicode.com/posts",
    params={"userId": 5}
)
posts5 = resp5.json()

# Fetch posts for user 7
resp7 = requests.get(
    "https://jsonplaceholder.typicode.com/posts",
    params={"userId": 7}
)
posts7 = resp7.json()

print(f"User 5 has {len(posts5)} posts")
print(f"User 7 has {len(posts7)} posts")

if len(posts5) > len(posts7):
    print("User 5 is more active.")
elif len(posts7) > len(posts5):
    print("User 7 is more active.")
else:
    print("Both users have the same number of posts.")


---

## 3.2 Working with Real APIs

Let's work with some real-world APIs.

### Question 3.5 (Solved): GitHub API - Public Repositories

The GitHub API is free to use (with rate limits) and doesn't require authentication for public data.

In [ ]:
# SOLVED EXAMPLE
import requests

# Fetch information about a popular repository
response = requests.get(
    "https://api.github.com/repos/pandas-dev/pandas",
    headers={"Accept": "application/vnd.github.v3+json"}
)

if response.ok:
    repo = response.json()
    print(f"Repository: {repo['full_name']}")
    print(f"Description: {repo['description']}")
    print(f"Stars: {repo['stargazers_count']:,}")
    print(f"Forks: {repo['forks_count']:,}")
    print(f"Language: {repo['language']}")
else:
    print(f"Error: {response.status_code}")

### Question 3.6: Compare Popular ML Libraries

Fetch information about these ML-related repositories and create a comparison table:
- `scikit-learn/scikit-learn`
- `pytorch/pytorch`
- `tensorflow/tensorflow`

Show: name, stars, forks, and primary language.

**Hint**: Loop through the repos and collect data into a list of dictionaries, then create a DataFrame.

In [ ]:
# YOUR CODE HERE
import requests
import pandas as pd

repos = [
    "scikit-learn/scikit-learn",
    "pytorch/pytorch",
    "tensorflow/tensorflow"
]

# Fetch data for each repo
data = []
for repo in repos:
    response = requests.get(
        f"https://api.github.com/repos/{repo}",
        headers={"Accept": "application/vnd.github.v3+json"}
    )
    if response.ok:
        r = response.json()
        data.append({
            "name":     r["full_name"],
            "stars":    r["stargazers_count"],
            "forks":    r["forks_count"],
            "language": r["language"]
        })
    else:
        print(f"Failed to fetch {repo}: {response.status_code}")
    time.sleep(0.5)   # respect rate limits

# Create a DataFrame
df_repos = pd.DataFrame(data)

# Display the comparison
print(df_repos.to_string(index=False))


### Question 3.7: Search GitHub Repositories

Use the GitHub search API to find the top 10 most starred repositories with "machine learning" in their description.

API endpoint: `https://api.github.com/search/repositories`

Parameters:
- `q`: search query (e.g., "machine learning")
- `sort`: "stars"
- `order`: "desc"
- `per_page`: 10

Print the name and star count of each repository.

In [ ]:
# YOUR CODE HERE
import requests

response = requests.get(
    "https://api.github.com/search/repositories",
    params={
        "q":        "machine learning",
        "sort":     "stars",
        "order":    "desc",
        "per_page": 10
    },
    headers={"Accept": "application/vnd.github.v3+json"}
)

if response.ok:
    results = response.json()
    print(f"Top {len(results['items'])} most-starred 'machine learning' repos:\n")
    for item in results["items"]:
        print(f"  ⭐ {item['stargazers_count']:>7,}  {item['full_name']}")
else:
    print(f"Request failed: {response.status_code}")


---

## 3.3 Error Handling

Real-world APIs fail. We need to handle errors gracefully.

### Question 3.8 (Solved): Handling HTTP Errors

In [ ]:
# SOLVED EXAMPLE
import requests

def fetch_with_error_handling(url):
    """Fetch URL with proper error handling."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()  # Raises exception for 4xx/5xx
        return response.json()
    except requests.exceptions.Timeout:
        print(f"Timeout: Request took too long")
    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error: {e.response.status_code}")
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
    return None

# Test with valid URL
print("Valid URL:")
data = fetch_with_error_handling("https://jsonplaceholder.typicode.com/posts/1")
if data:
    print(f"  Got post: {data['title'][:50]}...")

# Test with invalid URL (404)
print("\nInvalid URL (404):")
fetch_with_error_handling("https://jsonplaceholder.typicode.com/posts/99999")

### Question 3.9: Robust Fetcher Function

Write a function `safe_fetch(url, max_retries=3)` that:

1. Attempts to fetch the URL
2. If it fails with a 5xx error, retries up to `max_retries` times
3. Waits 1 second between retries
4. Returns the JSON data if successful, None otherwise

Test it with `https://httpbin.org/status/500` (always returns 500) and `https://jsonplaceholder.typicode.com/posts/1` (always works).

In [ ]:
# YOUR CODE HERE
import time
import requests

def safe_fetch(url, max_retries=3):
    """Fetch URL with retry logic for server errors."""
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, timeout=10)

            if response.ok:
                return response.json()

            # Retry only on 5xx (server) errors
            if 500 <= response.status_code < 600:
                print(f"  Attempt {attempt}/{max_retries} failed "
                      f"(HTTP {response.status_code}). Retrying in 1 s...")
                time.sleep(1)
                continue

            # 4xx errors are client errors — no point retrying
            print(f"  Client error: HTTP {response.status_code}")
            return None

        except requests.exceptions.Timeout:
            print(f"  Attempt {attempt}/{max_retries}: Timeout. Retrying in 1 s...")
            time.sleep(1)
        except requests.exceptions.RequestException as e:
            print(f"  Request error: {e}")
            return None

    print(f"  All {max_retries} attempts failed.")
    return None

# Test your function
print("Testing with working URL:")
result = safe_fetch("https://jsonplaceholder.typicode.com/posts/1")
print(f"Result: {result}")

print("\nTesting with failing URL (500):")
result = safe_fetch("https://httpbin.org/status/500")
print(f"Result: {result}")


---

# Part 4: The OMDb Movie API

Now let's work with the OMDb API - our main data source for the Netflix project.

**Note**: You need an API key from https://www.omdbapi.com/apikey.aspx (free tier available).

For this lab, we'll use a demo key that has limited functionality.

In [ ]:
# Set your API key here
# Get a free key from: https://www.omdbapi.com/apikey.aspx
OMDB_API_KEY = "YOUR_API_KEY_HERE"  # Replace with your actual key

# For demo purposes, you can try with key "demo" but it's very limited
# OMDB_API_KEY = "demo"

### Question 4.1 (Solved): Fetch a Single Movie

In [ ]:
# SOLVED EXAMPLE
import requests

def fetch_movie(title, year=None, api_key=OMDB_API_KEY):
    """Fetch movie data from OMDb API."""
    params = {
        "apikey": api_key,
        "t": title,  # Search by title
        "type": "movie"
    }
    if year:
        params["y"] = year
    
    response = requests.get("https://www.omdbapi.com/", params=params)
    
    if response.ok:
        data = response.json()
        if data.get("Response") == "True":
            return data
        else:
            print(f"Movie not found: {data.get('Error')}")
    return None

# Fetch Inception
movie = fetch_movie("Inception", 2010)
if movie:
    print(f"Title: {movie['Title']}")
    print(f"Year: {movie['Year']}")
    print(f"Director: {movie['Director']}")
    print(f"IMDB Rating: {movie['imdbRating']}")
    print(f"Genre: {movie['Genre']}")

### Question 4.2: Explore the Response

Fetch data for "The Dark Knight" and print ALL available fields in the response.

Which fields might be useful for predicting movie success?

In [ ]:
# YOUR CODE HERE
# Note: Replace OMDB_API_KEY with your actual key to get live data.
# Below we demonstrate the structure using a realistic mock response.

import json

# Simulated OMDb response for "The Dark Knight" (mirrors real API format)
mock_response = {
    "Title": "The Dark Knight",
    "Year": "2008",
    "Rated": "PG-13",
    "Released": "18 Jul 2008",
    "Runtime": "152 min",
    "Genre": "Action, Crime, Drama",
    "Director": "Christopher Nolan",
    "Writer": "Jonathan Nolan, Christopher Nolan",
    "Actors": "Christian Bale, Heath Ledger, Aaron Eckhart",
    "Plot": "When the menace known as the Joker wreaks havoc on Gotham City...",
    "Language": "English, Mandarin",
    "Country": "United States, United Kingdom",
    "Awards": "Won 2 Oscars. 159 wins & 163 nominations total",
    "Poster": "https://m.media-amazon.com/images/...",
    "Ratings": [
        {"Source": "Internet Movie Database", "Value": "9.0/10"},
        {"Source": "Rotten Tomatoes",         "Value": "94%"},
        {"Source": "Metacritic",              "Value": "84/100"}
    ],
    "Metascore": "84",
    "imdbRating": "9.0",
    "imdbVotes": "2,834,000",
    "imdbID": "tt0468569",
    "Type": "movie",
    "DVD": "09 Dec 2008",
    "BoxOffice": "$534,858,444",
    "Production": "N/A",
    "Website": "N/A",
    "Response": "True"
}

print("=== All fields in The Dark Knight response ===\n")
for field, value in mock_response.items():
    print(f"  {field:<14}: {value}")

print("\n=== Fields useful for predicting movie success ===")
useful = {
    "imdbRating":  "Target variable / quality proxy",
    "imdbVotes":   "Popularity / audience size",
    "BoxOffice":   "Revenue (key success metric)",
    "Genre":       "Content category — important feature",
    "Director":    "Track-record signal",
    "Runtime":     "Correlates with budget & engagement",
    "Rated":       "Audience restriction (PG-13 vs R)",
    "Metascore":   "Critical reception",
    "Awards":      "Prestige signal",
    "Released":    "Seasonality / competition context",
}
for f, reason in useful.items():
    print(f"  {f:<14}: {reason}")


### Question 4.3: Fetch Multiple Movies

Create a function `fetch_movies(titles)` that:
1. Takes a list of movie titles
2. Fetches data for each movie
3. Returns a list of movie dictionaries (only successful fetches)
4. Adds a 0.5 second delay between requests (to respect rate limits)

Test it with: `["Inception", "The Matrix", "Interstellar", "NonExistentMovie123"]`

In [ ]:
# YOUR CODE HERE
import requests
import time

def fetch_movies(titles, api_key=OMDB_API_KEY):
    """Fetch multiple movies from OMDb API."""
    results = []
    for title in titles:
        try:
            response = requests.get(
                "https://www.omdbapi.com/",
                params={"apikey": api_key, "t": title, "type": "movie"},
                timeout=10
            )
            if response.ok:
                data = response.json()
                if data.get("Response") == "True":
                    results.append(data)
                else:
                    print(f"  Not found: {title} — {data.get('Error', 'Unknown error')}")
            else:
                print(f"  HTTP {response.status_code} for: {title}")
        except requests.exceptions.RequestException as e:
            print(f"  Request error for {title}: {e}")
        time.sleep(0.5)   # respect rate limits
    return results

# Test
test_titles = ["Inception", "The Matrix", "Interstellar", "NonExistentMovie123"]
movies = fetch_movies(test_titles)
print(f"Successfully fetched {len(movies)} out of {len(test_titles)} movies")
for m in movies:
    print(f"  {m['Title']} ({m['Year']}) — IMDb: {m.get('imdbRating', 'N/A')}")


### Question 4.4: Create a Movie DataFrame

Using the movies you fetched, create a pandas DataFrame with these columns:
- title
- year (as integer)
- genre
- director
- imdb_rating (as float)
- imdb_votes (as integer, remove commas)
- runtime_minutes (as integer, extract from "148 min")
- box_office (keep as string for now)

**Hint**: You'll need to clean the data types.

In [ ]:
# YOUR CODE HERE
import pandas as pd
import re

# Use mock data so the code runs without an API key
mock_movies = [
    {"Title":"Inception",    "Year":"2010","Genre":"Action, Adventure, Sci-Fi",
     "Director":"Christopher Nolan","imdbRating":"8.8","imdbVotes":"2,400,000",
     "Runtime":"148 min","BoxOffice":"$292,576,195","Response":"True"},
    {"Title":"The Matrix",   "Year":"1999","Genre":"Action, Sci-Fi",
     "Director":"Lana Wachowski","imdbRating":"8.7","imdbVotes":"1,900,000",
     "Runtime":"136 min","BoxOffice":"$171,479,930","Response":"True"},
    {"Title":"Interstellar", "Year":"2014","Genre":"Adventure, Drama, Sci-Fi",
     "Director":"Christopher Nolan","imdbRating":"8.6","imdbVotes":"1,800,000",
     "Runtime":"169 min","BoxOffice":"$188,020,017","Response":"True"},
]

def build_movie_df(movie_list):
    rows = []
    for m in movie_list:
        # runtime_minutes: "148 min" -> 148
        runtime_str = m.get("Runtime", "N/A")
        runtime_match = re.search(r"(\d+)", runtime_str)
        runtime_min = int(runtime_match.group(1)) if runtime_match else None

        # imdb_votes: "2,400,000" -> 2400000
        votes_str = m.get("imdbVotes", "N/A").replace(",", "")
        imdb_votes = int(votes_str) if votes_str.isdigit() else None

        rows.append({
            "title":           m.get("Title"),
            "year":            int(m.get("Year", 0)[:4]),
            "genre":           m.get("Genre"),
            "director":        m.get("Director"),
            "imdb_rating":     float(m["imdbRating"]) if m.get("imdbRating", "N/A") != "N/A" else None,
            "imdb_votes":      imdb_votes,
            "runtime_minutes": runtime_min,
            "box_office":      m.get("BoxOffice", "N/A"),
        })
    return pd.DataFrame(rows)

df_movies = build_movie_df(mock_movies)
print(df_movies.to_string(index=False))
print(f"\nData types:\n{df_movies.dtypes}")


### Question 4.5: Search Movies by Title

OMDb also has a search endpoint that returns multiple results.

Use the `s` parameter instead of `t` to search for movies containing "Star Wars".

API endpoint: `https://www.omdbapi.com/?apikey=YOUR_KEY&s=Star Wars&type=movie`

Print the title and year of each result.

In [ ]:
# YOUR CODE HERE
import requests

def search_movies_by_title(query, movie_type="movie", api_key=OMDB_API_KEY):
    """Search OMDb for movies matching a query string."""
    response = requests.get(
        "https://www.omdbapi.com/",
        params={"apikey": api_key, "s": query, "type": movie_type},
        timeout=10
    )
    if not response.ok:
        print(f"HTTP error: {response.status_code}")
        return []
    data = response.json()
    if data.get("Response") != "True":
        print(f"Search failed: {data.get('Error', 'Unknown error')}")
        return []
    return data.get("Search", [])

results = search_movies_by_title("Star Wars")
print(f"Found {len(results)} results for 'Star Wars':\n")
for movie in results:
    print(f"  {movie['Year']}  {movie['Title']}")


### Question 4.6: Handle Pagination

The OMDb search API returns 10 results per page and includes a `totalResults` field.

Write a function `search_all_movies(query)` that:
1. Searches for movies matching the query
2. Fetches ALL pages of results (use the `page` parameter)
3. Returns a list of all movies found

**Hint**: `totalResults` tells you how many movies exist. Divide by 10 to get the number of pages.

Test with a query that has many results like "Batman".

In [ ]:
# YOUR CODE HERE
import requests
import math
import time

def search_all_movies(query, api_key=OMDB_API_KEY):
    """Search OMDb and return ALL matching movies across all pages."""
    all_movies = []

    # First request to get total count
    first_resp = requests.get(
        "https://www.omdbapi.com/",
        params={"apikey": api_key, "s": query, "type": "movie", "page": 1},
        timeout=10
    )
    if not first_resp.ok:
        print(f"HTTP error: {first_resp.status_code}")
        return []

    first_data = first_resp.json()
    if first_data.get("Response") != "True":
        print(f"Search failed: {first_data.get('Error', 'Unknown error')}")
        return []

    total_results = int(first_data.get("totalResults", 0))
    total_pages   = math.ceil(total_results / 10)
    all_movies.extend(first_data.get("Search", []))

    print(f"Total results: {total_results}  |  Pages: {total_pages}")

    # Fetch remaining pages
    for page in range(2, total_pages + 1):
        resp = requests.get(
            "https://www.omdbapi.com/",
            params={"apikey": api_key, "s": query, "type": "movie", "page": page},
            timeout=10
        )
        if resp.ok and resp.json().get("Response") == "True":
            all_movies.extend(resp.json().get("Search", []))
        time.sleep(0.3)   # polite pacing

    return all_movies

# Test
all_batman = search_all_movies("Batman")
print(f"Found {len(all_batman)} Batman movies")
if all_batman:
    for m in all_batman[:5]:
        print(f"  {m['Year']}  {m['Title']}")


---

# Part 5: Web Scraping with BeautifulSoup

When APIs don't exist or don't have what we need, we scrape.

## 5.1 HTML Basics

### Question 5.1 (Solved): Parse HTML

In [ ]:
# SOLVED EXAMPLE
from bs4 import BeautifulSoup

html = """
<html>
<body>
    <div class="movie" id="movie-1">
        <h2 class="title">Inception</h2>
        <span class="year">2010</span>
        <span class="rating">8.8</span>
        <a href="/movies/inception">More Info</a>
    </div>
    <div class="movie" id="movie-2">
        <h2 class="title">The Matrix</h2>
        <span class="year">1999</span>
        <span class="rating">8.7</span>
        <a href="/movies/matrix">More Info</a>
    </div>
</body>
</html>
"""

soup = BeautifulSoup(html, 'html.parser')

# Find all movie divs
movies = soup.find_all('div', class_='movie')
print(f"Found {len(movies)} movies\n")

# Extract data from each
for movie in movies:
    title = movie.find('h2', class_='title').text
    year = movie.find('span', class_='year').text
    rating = movie.find('span', class_='rating').text
    link = movie.find('a')['href']
    
    print(f"{title} ({year}) - Rating: {rating} - Link: {link}")

### Question 5.2: CSS Selectors

Rewrite the above extraction using CSS selectors (`.select()` and `.select_one()`) instead of `.find()` and `.find_all()`.

**Hint**: 
- `.movie` selects elements with class "movie"
- `.movie .title` selects elements with class "title" inside class "movie"

In [ ]:
# YOUR CODE HERE
from bs4 import BeautifulSoup

# Re-create the same HTML used in Q5.1
html = """
<html><body>
    <div class="movie" id="movie-1">
        <h2 class="title">Inception</h2>
        <span class="year">2010</span>
        <span class="rating">8.8</span>
        <a href="/movies/inception">More Info</a>
    </div>
    <div class="movie" id="movie-2">
        <h2 class="title">The Matrix</h2>
        <span class="year">1999</span>
        <span class="rating">8.7</span>
        <a href="/movies/matrix">More Info</a>
    </div>
</body></html>
"""

soup = BeautifulSoup(html, "html.parser")

# Extract using CSS selectors
print("Using CSS selectors:")
for movie in soup.select(".movie"):          # all elements with class "movie"
    title  = movie.select_one(".title").text
    year   = movie.select_one(".year").text
    rating = movie.select_one(".rating").text
    link   = movie.select_one("a")["href"]
    print(f"  Title: {title}, Year: {year}, Rating: {rating}, Link: {link}")


### Question 5.3: Scrape a Real Website

Let's scrape the example website `http://quotes.toscrape.com/` which is designed for scraping practice.

Extract all quotes from the first page, including:
- The quote text
- The author name
- The tags

Return the results as a list of dictionaries.

In [ ]:
# YOUR CODE HERE
import requests
from bs4 import BeautifulSoup

# Fetch the page
url = "http://quotes.toscrape.com/"
response = requests.get(url, timeout=10)

# Parse the HTML
soup = BeautifulSoup(response.text, "html.parser")

# Extract quotes
quotes = []
for block in soup.select("div.quote"):
    text   = block.select_one("span.text").text.strip('\u201c\u201d"')
    author = block.select_one("small.author").text
    tags   = [tag.text for tag in block.select("a.tag")]
    quotes.append({"text": text, "author": author, "tags": tags})

# Print results
print(f"Scraped {len(quotes)} quotes from page 1:\n")
for i, q in enumerate(quotes, 1):
    print(f"[{i}] \"{q['text'][:70]}...\"")
    print(f"    — {q['author']}  |  tags: {', '.join(q['tags'])}\n")


### Question 5.4: Handle Pagination in Scraping

The quotes website has multiple pages. Scrape the first 3 pages and collect all quotes.

Pages follow the pattern:
- Page 1: `http://quotes.toscrape.com/page/1/`
- Page 2: `http://quotes.toscrape.com/page/2/`
- etc.

**Remember**: Add a delay between requests to be polite!

In [ ]:
# YOUR CODE HERE
import requests
from bs4 import BeautifulSoup
import time

all_quotes = []

for page_num in range(1, 4):   # pages 1, 2, 3
    url = f"http://quotes.toscrape.com/page/{page_num}/"
    response = requests.get(url, timeout=10)
    soup     = BeautifulSoup(response.text, "html.parser")

    page_quotes = []
    for block in soup.select("div.quote"):
        text   = block.select_one("span.text").text.strip('\u201c\u201d"')
        author = block.select_one("small.author").text
        tags   = [tag.text for tag in block.select("a.tag")]
        page_quotes.append({"text": text, "author": author, "tags": tags})

    all_quotes.extend(page_quotes)
    print(f"Page {page_num}: scraped {len(page_quotes)} quotes")
    time.sleep(0.5)   # be polite

print(f"\nTotal quotes collected: {len(all_quotes)}")
print("\nFirst 3 quotes:")
for q in all_quotes[:3]:
    print(f"  '{q['text'][:60]}...' — {q['author']}")


### Question 5.5: Extract Table Data

Scrape the table from `https://www.w3schools.com/html/html_tables.asp`.

The table contains company data. Extract all rows and create a pandas DataFrame.

**Hint**: Look for `<table>`, `<tr>` (table row), `<th>` (header), and `<td>` (data cell) elements.

In [ ]:
# YOUR CODE HERE
import requests
from bs4 import BeautifulSoup
import pandas as pd

# --- Manual approach ---
url = "https://www.w3schools.com/html/html_tables.asp"
response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
soup     = BeautifulSoup(response.text, "html.parser")

# Locate the first table on the page
table = soup.find("table")

# Extract headers
headers = [th.text.strip() for th in table.find_all("th")]
print(f"Headers: {headers}")

# Extract rows
rows = []
for tr in table.find("tbody").find_all("tr") if table.find("tbody") else table.find_all("tr")[1:]:
    cells = [td.text.strip() for td in tr.find_all("td")]
    if cells:
        rows.append(cells)

df_table = pd.DataFrame(rows, columns=headers if headers else None)
print("\nManually scraped table:")
print(df_table.to_string(index=False))

# --- Automatic approach with pandas ---
print("\n--- pandas.read_html() approach ---")
tables = pd.read_html(url)
print(f"Found {len(tables)} table(s) on the page")
print(tables[0].to_string(index=False))


---

# Part 6: Building the Movie Data Pipeline

Now let's put everything together to build a complete data collection pipeline for our Netflix project.

## 6.1 The Complete Pipeline

### Question 6.1 (Solved): Movie Data Collector Class

In [ ]:
# SOLVED EXAMPLE
import requests
import pandas as pd
import time
from typing import List, Dict, Optional

class MovieDataCollector:
    """Collect movie data from OMDb API."""
    
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "http://www.omdbapi.com/"
        self.delay = 0.5  # Seconds between requests
    
    def fetch_movie(self, title: str, year: Optional[int] = None) -> Optional[Dict]:
        """Fetch a single movie by title."""
        params = {
            "apikey": self.api_key,
            "t": title,
            "type": "movie"
        }
        if year:
            params["y"] = year
        
        try:
            response = requests.get(self.base_url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
            
            if data.get("Response") == "True":
                return data
        except Exception as e:
            print(f"Error fetching {title}: {e}")
        
        return None
    
    def fetch_movies(self, titles: List[str]) -> List[Dict]:
        """Fetch multiple movies."""
        movies = []
        
        for i, title in enumerate(titles):
            print(f"Fetching {i+1}/{len(titles)}: {title}")
            movie = self.fetch_movie(title)
            
            if movie:
                movies.append(movie)
            
            time.sleep(self.delay)
        
        return movies
    
    def to_dataframe(self, movies: List[Dict]) -> pd.DataFrame:
        """Convert movie data to cleaned DataFrame."""
        if not movies:
            return pd.DataFrame()
        
        # Extract relevant fields
        rows = []
        for m in movies:
            rows.append({
                "title": m.get("Title"),
                "year": m.get("Year"),
                "genre": m.get("Genre"),
                "director": m.get("Director"),
                "actors": m.get("Actors"),
                "imdb_rating": m.get("imdbRating"),
                "imdb_votes": m.get("imdbVotes"),
                "runtime": m.get("Runtime"),
                "box_office": m.get("BoxOffice"),
                "imdb_id": m.get("imdbID")
            })
        
        df = pd.DataFrame(rows)
        
        # Clean data types
        df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
        df["imdb_rating"] = pd.to_numeric(df["imdb_rating"], errors="coerce")
        df["imdb_votes"] = df["imdb_votes"].str.replace(",", "").pipe(pd.to_numeric, errors="coerce").astype("Int64")
        # Fix: str.extract returns a DataFrame, we need column 0 to get a Series
        df["runtime_min"] = df["runtime"].str.extract(r"(\d+)")[0].pipe(pd.to_numeric, errors="coerce").astype("Int64")
        
        return df

# Usage example
# collector = MovieDataCollector(OMDB_API_KEY)
# movies = collector.fetch_movies(["Inception", "The Matrix"])
# df = collector.to_dataframe(movies)
# print(df)

### Question 6.2: Add Search Functionality

Extend the `MovieDataCollector` class to add a `search_movies(query, max_results=50)` method that:
1. Searches for movies matching the query
2. Handles pagination to get up to `max_results` movies
3. For each search result, fetches the full movie details
4. Returns the detailed movie data

**Hint**: Search results only contain basic info (title, year, poster, imdbID). You need to use the imdbID to fetch full details.

In [ ]:
# YOUR CODE HERE
import requests
import math
import time
from typing import List, Dict, Optional

class MovieDataCollector:
    """Collect movie data from OMDb API (extended with search functionality)."""

    def __init__(self, api_key: str):
        self.api_key  = api_key
        self.base_url = "http://www.omdbapi.com/"
        self.delay    = 0.5

    def fetch_movie(self, title: str, year: Optional[int] = None) -> Optional[Dict]:
        """Fetch a single movie by title."""
        params: Dict = {"apikey": self.api_key, "t": title, "type": "movie"}
        if year:
            params["y"] = year
        try:
            resp = requests.get(self.base_url, params=params, timeout=10)
            if resp.ok:
                data = resp.json()
                return data if data.get("Response") == "True" else None
        except requests.exceptions.RequestException:
            pass
        return None

    def fetch_by_id(self, imdb_id: str) -> Optional[Dict]:
        """Fetch full movie details using an IMDb ID."""
        try:
            resp = requests.get(
                self.base_url,
                params={"apikey": self.api_key, "i": imdb_id, "type": "movie"},
                timeout=10
            )
            if resp.ok:
                data = resp.json()
                return data if data.get("Response") == "True" else None
        except requests.exceptions.RequestException:
            pass
        return None

    def search_movies(self, query: str, max_results: int = 50) -> List[Dict]:
        """
        Search for movies matching query.
        Handles pagination and enriches each result with full details.
        """
        found_basic  = []
        page         = 1
        total_pages  = 1   # will be updated after first response

        # ── Step 1: Collect basic search results (paginated) ──────────────
        while page <= total_pages and len(found_basic) < max_results:
            resp = requests.get(
                self.base_url,
                params={"apikey": self.api_key, "s": query,
                        "type": "movie", "page": page},
                timeout=10
            )
            time.sleep(self.delay)

            if not resp.ok:
                print(f"HTTP error on page {page}: {resp.status_code}")
                break

            data = resp.json()
            if data.get("Response") != "True":
                print(f"Search error: {data.get('Error', 'Unknown')}")
                break

            if page == 1:
                total_results = int(data.get("totalResults", 0))
                total_pages   = min(math.ceil(total_results / 10),
                                    math.ceil(max_results / 10))

            found_basic.extend(data.get("Search", []))
            page += 1

        found_basic = found_basic[:max_results]

        # ── Step 2: Enrich each result with full details via imdbID ───────
        detailed = []
        for basic in found_basic:
            full = self.fetch_by_id(basic["imdbID"])
            time.sleep(self.delay)
            if full:
                detailed.append(full)

        print(f"search_movies('{query}'): {len(detailed)} detailed results returned")
        return detailed


# Quick smoke test (won't make real API calls without a valid key)
collector = MovieDataCollector(OMDB_API_KEY)
print("MovieDataCollector with search_movies() is ready.")
print("Call collector.search_movies('Inception') with a valid API key to fetch real data.")


### Question 6.3: Build a Genre-Based Dataset

Use your collector to build a dataset of popular movies from different genres:

1. Search for 10 movies each for: "action", "comedy", "drama", "horror", "sci-fi"
2. Combine all results into a single DataFrame
3. Remove any duplicates (some movies might appear in multiple searches)
4. Save to CSV

**Note**: This might take a while due to rate limiting. Start with fewer movies for testing.

In [ ]:
# YOUR CODE HERE
import pandas as pd
import time, re

# ── If you have a valid API key, uncomment the block below ──────────────────
# collector  = MovieDataCollector(OMDB_API_KEY)
# genres     = ["action", "comedy", "drama", "horror", "sci-fi"]
# all_movies = []
# for genre in genres:
#     print(f"Searching '{genre}'...")
#     results = collector.search_movies(genre, max_results=10)
#     for m in results:
#         m["search_genre"] = genre
#     all_movies.extend(results)
#     time.sleep(1)
# df_genre = pd.DataFrame(all_movies).drop_duplicates(subset="imdbID")
# df_genre.to_csv("movies_by_genre.csv", index=False)
# print(f"Saved {len(df_genre)} unique movies to movies_by_genre.csv")
# ─────────────────────────────────────────────────────────────────────────────

# Demo with realistic mock data (runs without an API key)
mock_genre_data = [
    {"Title":"Mad Max: Fury Road","Year":"2015","Genre":"Action, Adventure, Sci-Fi",
     "Director":"George Miller","imdbRating":"8.1","imdbVotes":"1,031,000",
     "Runtime":"120 min","BoxOffice":"$45,885,504","imdbID":"tt1392190","search_genre":"action"},
    {"Title":"The Grand Budapest Hotel","Year":"2014","Genre":"Adventure, Comedy, Crime",
     "Director":"Wes Anderson","imdbRating":"8.1","imdbVotes":"800,000",
     "Runtime":"99 min","BoxOffice":"$59,301,324","imdbID":"tt2278388","search_genre":"comedy"},
    {"Title":"12 Years a Slave","Year":"2013","Genre":"Biography, Drama, History",
     "Director":"Steve McQueen","imdbRating":"8.1","imdbVotes":"700,000",
     "Runtime":"134 min","BoxOffice":"$56,671,993","imdbID":"tt2024544","search_genre":"drama"},
    {"Title":"Get Out","Year":"2017","Genre":"Horror, Mystery, Thriller",
     "Director":"Jordan Peele","imdbRating":"7.7","imdbVotes":"650,000",
     "Runtime":"104 min","BoxOffice":"$176,040,665","imdbID":"tt5052448","search_genre":"horror"},
    {"Title":"Arrival","Year":"2016","Genre":"Drama, Mystery, Sci-Fi",
     "Director":"Denis Villeneuve","imdbRating":"7.9","imdbVotes":"590,000",
     "Runtime":"116 min","BoxOffice":"$100,546,139","imdbID":"tt2543164","search_genre":"sci-fi"},
]

def clean_movie(m):
    votes_str = m.get("imdbVotes","N/A").replace(",","")
    rm = re.search(r"(\d+)", m.get("Runtime","0"))
    return {
        "title":        m["Title"],
        "year":         int(m["Year"][:4]),
        "genre":        m["Genre"],
        "director":     m["Director"],
        "imdb_rating":  float(m["imdbRating"]) if m.get("imdbRating","N/A")!="N/A" else None,
        "imdb_votes":   int(votes_str) if votes_str.isdigit() else None,
        "runtime_min":  int(rm.group(1)) if rm else None,
        "box_office":   m.get("BoxOffice","N/A"),
        "imdb_id":      m.get("imdbID"),
        "search_genre": m.get("search_genre"),
    }

df_genre = pd.DataFrame([clean_movie(m) for m in mock_genre_data])
df_genre = df_genre.drop_duplicates(subset="imdb_id")
df_genre.to_csv("movies_by_genre.csv", index=False)
print(f"Dataset: {len(df_genre)} unique movies")
print(df_genre.to_string(index=False))


### Question 6.4: Data Quality Analysis

Using the dataset you created:

1. How many movies have missing IMDB ratings?
2. How many movies have missing box office data?
3. What's the distribution of ratings? (min, max, mean, median)
4. Which directors appear most frequently?
5. What's the average runtime by genre?

These quality checks will be important for Week 2 (Data Validation)!

In [ ]:
# YOUR CODE HERE
import pandas as pd

# Load the dataset saved in Q6.3
try:
    df = pd.read_csv("movies_by_genre.csv")
except FileNotFoundError:
    print("Run Q6.3 first to generate the CSV.")
    df = pd.DataFrame()   # empty fallback

if not df.empty:
    # 1. Missing IMDb ratings
    missing_ratings = df["imdb_rating"].isna().sum()
    print(f"1. Movies with missing IMDb ratings : {missing_ratings}")

    # 2. Missing box office data
    missing_bo = (df["box_office"].isna() | (df["box_office"] == "N/A")).sum()
    print(f"2. Movies with missing box office   : {missing_bo}")

    # 3. Rating distribution
    print("\n3. IMDb rating distribution:")
    print(f"   Min    : {df['imdb_rating'].min()}")
    print(f"   Max    : {df['imdb_rating'].max()}")
    print(f"   Mean   : {df['imdb_rating'].mean():.2f}")
    print(f"   Median : {df['imdb_rating'].median():.2f}")

    # 4. Most frequent directors
    print("\n4. Most frequent directors:")
    print(df["director"].value_counts().head(5).to_string())

    # 5. Average runtime by search genre
    print("\n5. Average runtime (min) by search genre:")
    print(df.groupby("search_genre")["runtime_min"].mean().round(1).to_string())


---

# Part 7: Challenge Problems

These are optional advanced exercises for those who finish early.

### Challenge 7.1: Rate Limit Handler

Create a `RateLimiter` class that:
1. Tracks how many requests have been made
2. Automatically adds delays to stay under a rate limit
3. Handles 429 (Too Many Requests) responses by waiting and retrying

```python
limiter = RateLimiter(requests_per_minute=30)
response = limiter.get("https://api.example.com/data")
```

In [ ]:
# YOUR CODE HERE
import time
import requests
from collections import deque

class RateLimiter:
    """
    A wrapper around requests.get() that enforces a per-minute rate limit
    and automatically retries on 429 (Too Many Requests) responses.
    """

    def __init__(self, requests_per_minute: int = 30):
        self.requests_per_minute = requests_per_minute
        self.min_interval = 60.0 / requests_per_minute   # seconds between requests
        self._timestamps: deque = deque()                  # sliding window

    def _wait_if_needed(self):
        """Block until we are within the rate limit window."""
        now = time.time()
        # Remove timestamps older than 60 s
        while self._timestamps and now - self._timestamps[0] > 60:
            self._timestamps.popleft()

        if len(self._timestamps) >= self.requests_per_minute:
            sleep_for = 60 - (now - self._timestamps[0])
            if sleep_for > 0:
                print(f"  [RateLimiter] Sleeping {sleep_for:.2f}s to stay under limit...")
                time.sleep(sleep_for)

    def get(self, url: str, **kwargs) -> requests.Response:
        """Make a GET request, respecting the rate limit and retrying on 429."""
        max_retries = kwargs.pop("max_retries", 3)

        for attempt in range(1, max_retries + 1):
            self._wait_if_needed()
            response = requests.get(url, **kwargs)
            self._timestamps.append(time.time())

            if response.status_code == 429:
                retry_after = int(response.headers.get("Retry-After", 5))
                print(f"  [RateLimiter] 429 received. Waiting {retry_after}s (attempt {attempt})...")
                time.sleep(retry_after)
                continue

            return response

        raise RuntimeError(f"Max retries ({max_retries}) exceeded for {url}")


# Quick demonstration
limiter = RateLimiter(requests_per_minute=30)
print("RateLimiter is ready. Example usage:")
print("  limiter = RateLimiter(requests_per_minute=30)")
print("  response = limiter.get('https://api.example.com/data')")
print("\nTracking up to 30 requests/min automatically.")


### Challenge 7.2: Async Movie Collector

The synchronous approach is slow because we wait for each request to complete.

Create an async version using `aiohttp` that can fetch multiple movies concurrently (while still respecting rate limits).

Compare the time to fetch 20 movies with sync vs async approach.

In [ ]:
# YOUR CODE HERE
# Hint: You'll need to install aiohttp: pip install aiohttp
# And use asyncio to run the async code

import asyncio
import time
import requests  # used for the synchronous baseline

# ── Synchronous baseline ─────────────────────────────────────────────────────
def fetch_sync(url: str) -> dict:
    resp = requests.get(url, timeout=10)
    return resp.json() if resp.ok else {}

def fetch_many_sync(urls):
    return [fetch_sync(u) for u in urls]

# ── Async version using aiohttp ───────────────────────────────────────────────
try:
    import aiohttp

    async def fetch_async(session, url: str) -> dict:
        async with session.get(url) as resp:
            return await resp.json() if resp.status == 200 else {}

    async def fetch_many_async(urls, concurrency: int = 5):
        sem = asyncio.Semaphore(concurrency)   # limit concurrent connections

        async def bounded_fetch(session, url):
            async with sem:
                return await fetch_async(session, url)

        async with aiohttp.ClientSession() as session:
            tasks = [bounded_fetch(session, u) for u in urls]
            return await asyncio.gather(*tasks)

    # ── Benchmark ──────────────────────────────────────────────────────────
    post_urls = [f"https://jsonplaceholder.typicode.com/posts/{i}" for i in range(1, 21)]

    # Sync
    t0 = time.time()
    results_sync  = fetch_many_sync(post_urls)
    t_sync        = time.time() - t0

    # Async
    t0 = time.time()
    results_async = asyncio.run(fetch_many_async(post_urls))
    t_async       = time.time() - t0

    print(f"Fetched {len(results_sync)} posts synchronously  in {t_sync:.2f}s")
    print(f"Fetched {len(results_async)} posts asynchronously in {t_async:.2f}s")
    print(f"Speedup: {t_sync / t_async:.1f}x")

except ImportError:
    print("aiohttp not installed. Install it with:  pip install aiohttp")
    print("Showing async code structure only.\n")
    print("""
async def fetch_async(session, url):
    async with session.get(url) as resp:
        return await resp.json()

async def fetch_many_async(urls, concurrency=5):
    sem = asyncio.Semaphore(concurrency)
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_async(session, u) for u in urls]
        return await asyncio.gather(*tasks)

results = asyncio.run(fetch_many_async(urls))
""")


### Challenge 7.3: Multi-Source Data Fusion

Create a data collection pipeline that:
1. Fetches basic movie data from OMDb
2. Enriches it with additional data from another source (e.g., Wikipedia API for plot summaries)
3. Merges the data based on movie title/year
4. Handles cases where data is missing from one source

Wikipedia API example:
```
https://en.wikipedia.org/api/rest_v1/page/summary/Inception_(film)
```

In [ ]:
# YOUR CODE HERE
import requests
import time
import pandas as pd
from typing import Optional, Dict

# ── Wikipedia helper ─────────────────────────────────────────────────────────
def fetch_wikipedia_summary(title: str) -> Optional[Dict]:
    """
    Fetch a Wikipedia page summary via the REST API.
    Tries '<Title>_(film)' first, then plain '<Title>'.
    """
    for query in [f"{title.replace(' ', '_')}_(film)", title.replace(' ', '_')]:
        url  = f"https://en.wikipedia.org/api/rest_v1/page/summary/{query}"
        resp = requests.get(url, timeout=10)
        if resp.ok and resp.json().get("type") != "disambiguation":
            data = resp.json()
            return {
                "wiki_title":   data.get("title"),
                "wiki_extract": data.get("extract", "")[:300],
                "wiki_url":     data.get("content_urls", {}).get("desktop", {}).get("page"),
            }
    return None

# ── OMDb helper (re-used from earlier) ───────────────────────────────────────
def fetch_omdb(title: str, api_key: str) -> Optional[Dict]:
    resp = requests.get(
        "https://www.omdbapi.com/",
        params={"apikey": api_key, "t": title, "type": "movie"},
        timeout=10
    )
    if resp.ok:
        data = resp.json()
        return data if data.get("Response") == "True" else None
    return None

# ── Fusion pipeline ──────────────────────────────────────────────────────────
def build_fused_dataset(titles, api_key: str) -> pd.DataFrame:
    """
    For each movie title:
      1. Fetch OMDb data
      2. Fetch Wikipedia summary
      3. Merge on title/year; mark missing sources
    """
    rows = []
    for title in titles:
        omdb = fetch_omdb(title, api_key)
        time.sleep(0.3)
        wiki = fetch_wikipedia_summary(title)
        time.sleep(0.3)

        row = {"title": title}
        if omdb:
            row.update({
                "year":        omdb.get("Year"),
                "imdb_rating": omdb.get("imdbRating"),
                "genre":       omdb.get("Genre"),
                "director":    omdb.get("Director"),
                "source_omdb": True,
            })
        else:
            row["source_omdb"] = False

        if wiki:
            row.update({
                "wiki_extract": wiki["wiki_extract"],
                "wiki_url":     wiki["wiki_url"],
                "source_wiki":  True,
            })
        else:
            row["source_wiki"] = False

        rows.append(row)
        print(f"  {title}: omdb={row['source_omdb']}, wiki={row['source_wiki']}")

    return pd.DataFrame(rows)


# Demo
titles = ["Inception", "The Matrix", "NonExistentFilmXYZ"]
print("Building fused dataset...")
df_fused = build_fused_dataset(titles, OMDB_API_KEY)
print("\nFused dataset:")
print(df_fused[["title","year","imdb_rating","source_omdb","source_wiki"]].to_string(index=False))


---

# Summary

In this lab, you learned:

1. **HTTP Fundamentals**: URLs, status codes, headers
2. **curl**: Command-line HTTP requests
3. **Python requests**: Programmatic data collection
4. **Error handling**: Timeouts, retries, status codes
5. **OMDb API**: Real-world movie data
6. **BeautifulSoup**: Web scraping when APIs don't exist
7. **Data pipelines**: Building reusable collection code

## Next Week

**Week 2: Data Validation & Quality**

The data we collected today is messy! Next week we'll learn:
- Schema validation with Pydantic
- Data type cleaning
- Handling missing values
- Quality metrics

---

## Submission

Save your completed notebook and submit:
1. This notebook with all cells executed
2. The CSV file of movies you collected
3. A brief summary (1 paragraph) of what you learned